# Pick and place a cube with a Franka Panda (Isaac Sim, real-robot mode)

The same task as `stretch_pick_place_cram.ipynb` — grasp a cube off one stand and
put it on another — with the hard parts of the Stretch taken out. No mobile base
to drive, no telescoping arm, and a hand whose fingers slide straight at each
other instead of swinging on an arc. What is left is the grasp itself.

As in the Stretch demo, **the cube is a real rigid body in Isaac Sim**. The
fingers have to close on it for real and friction has to hold it. The sim
publishes the cube's true pose alongside, so `cube_status()` can print what CRAM
*believes* against what the physics actually did; a grasp that misses shows up
as the two drifting apart.

**Kernel**: select **CRAM**.

## Scene

An empty world: ground plane, the Panda bolted to the origin, two stands. The
Panda's base *is* the `map` frame, so every number below is in robot coordinates.

```
        y
        ^
  +0.2  |            [place stand]
        |
   0.0  |  (Panda)
        |
  -0.2  |            [pick stand + cube]
        +--------------------------------> x
          0.0          0.45
```

Both stands are 0.49 m from the base — well inside the 0.85 m reach and far
enough out that the arm is not folded back on itself. Their tops are at
z = 0.30, the cube centre at z = 0.325.

## How the gripper is told to come at the cube

CRAM's grasp maths assumes a tool frame whose **x-axis is the approach axis**.
The Panda's hand points its **z-axis** out between the fingers, so
`PandaGripper.front_facing_orientation` carries the 90° rotation about y that
reconciles the two (see `cram_vrb_lab/robots/panda/semantic_model.py`). With that
in place the standard grasp descriptions mean what they say, and

    ApproachDirection.FRONT + VerticalAlignment.TOP

is a straight top-down grasp: the hand descends along map −z with the fingers
closing along map y. That is the natural grasp for something standing on a table
and the one used below.

## Start the simulation and giskard server

Separate scripts from the Stretch demos — a different robot, a different scene
and a different set of topics — so both sets can coexist. Nothing here needs
RViz: Isaac's own viewport already shows the arm and the props.

In [1]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parent  # this notebook lives in demos/
sys.path.insert(0, str(REPO))

# import os
# os.environ["DISPLAY"] = ":0"

from launcher import (
    PANDA_SERVER_SCRIPT,
    PANDA_SIM_MARKER,
    PANDA_SIM_SCRIPT,
    start_giskard_server,
    start_isaac_sim,
    stop,
)

sim_proc = start_isaac_sim(sim_script=PANDA_SIM_SCRIPT, marker=PANDA_SIM_MARKER)
giskard_proc = start_giskard_server(server_script=PANDA_SERVER_SCRIPT)

starting isaac sim, logging to /tmp/isaac_sim.log
isaac sim ready after 18s
starting giskard server, logging to /tmp/giskard_server.log
giskard server ready after 8s


## Connect and build a CRAM `Context`

Same as the Stretch notebooks: fetch the world the giskard server built over its
`fetch_world` service, keep it live with a `WorldSynchronizer`, and wrap it in a
`Context`.

The Stretch's `alternative_motion_mappings` do not apply — they exist to give a
mobile base better motions, and there is no base here; CRAM's default
`MoveToolCenterPointMotion`, a plain `CartesianPose` on the tool frame, is
exactly right. The Panda contributes one mapping of its own,
`PandaMoveGripper`, which is what stops a grasp from hanging forever: see step 3.

In [2]:
import threading

import nest_asyncio
import rclpy
from rclpy.executors import MultiThreadedExecutor

nest_asyncio.apply()  # CRAM's REAL execution calls GiskardWrapper.execute,
                      # which run_until_completes inside the already-running kernel loop.

from coraplex.datastructures.dataclasses import Context
from semantic_digital_twin.adapters.ros.world_fetcher import fetch_world_from_service
from semantic_digital_twin.adapters.ros.world_synchronizer import WorldSynchronizer

from cram_vrb_lab.robots.panda.motions import PANDA_MOTION_MAPPINGS
from cram_vrb_lab.robots.panda.semantic_model import Panda

if not rclpy.ok():
    rclpy.init()
node = rclpy.create_node('cram_panda_node')
executor = MultiThreadedExecutor()
executor.add_node(node)
threading.Thread(target=executor.spin, daemon=True, name='rclpy-executor').start()

world = fetch_world_from_service(node=node, timeout_seconds=300)
WorldSynchronizer(_world=world, node=node)

robot = world.get_semantic_annotations_by_type(Panda)
robot = robot[0] if robot else Panda.from_world(world)

context = Context(
    world=world,
    robot=robot,
    ros_node=node,
    evaluate_conditions=False,
    alternative_motion_mappings=PANDA_MOTION_MAPPINGS,
)
print('connected, robot:', type(robot).__name__)

connected, robot: Panda


## A small run helper

`with real_robot(...)` sets the execution type to REAL, so `plan.perform()`
builds each action's giskard motion and streams it to the running server.

In [3]:
from coraplex.execution_environment import real_robot
from coraplex.plans.factories import execute_single, sequential


def run_plan(plan, collision_avoidance=True):
    """Perform a CRAM plan on the real (sim) robot via giskard."""
    with real_robot(collision_avoidance=collision_avoidance):
        plan.perform()
    print('done')

## 1. Put the props into the digital twin

Isaac already has the cube and the stands as physics bodies. CRAM plans against
the twin, so the same three boxes have to exist there too — `add_props_to_twin`
builds them from the shared `PANDA_LAYOUT`. The change goes out over
`/world_sync`, so the giskard server's copy of the world learns about them and
can avoid the stands.

`sync_cube_from_sim` then snaps the twin's cube onto the pose Isaac's physics
actually settled it at — a one-shot perception stand-in, and the only channel
through which the twin ever learns the cube is not where CRAM assumed.

In [4]:
import numpy as np

from cram_vrb_lab.scenes.props import constants as props
from cram_vrb_lab.scenes.props.constants import PANDA_LAYOUT
from cram_vrb_lab.scenes.props.twin_props import (
    CubePoseSensor,
    add_props_to_twin,
    sync_cube_from_sim,
)

cube = add_props_to_twin(world, layout=PANDA_LAYOUT)
cube_sensor = CubePoseSensor(node)

print('twin cube:', np.round(np.asarray(cube.global_pose.to_np())[:3, 3], 3))
print('sim cube: ', np.round(sync_cube_from_sim(world, cube, cube_sensor), 3))

twin cube: [ 0.45  -0.2    0.325]
sim cube:  [ 0.45  -0.2    0.325]


### Believed vs. actual

The one measurement this notebook is built around. CRAM's `AttachNode` moves the
cube along with the gripper *in the twin* the moment the close-gripper motion
finishes — whether or not the physical fingers caught anything. So the twin
always reports a successful grasp. Isaac does not.

While the cube is held, a gap of a centimetre or two is normal (the twin freezes
it at the tool frame; the real cube sits wherever the fingers gripped it). A gap
that keeps growing means the cube was left behind or dropped.

In [5]:
def cube_status(label=''):
    """Print where CRAM believes the cube is vs where Isaac's physics has it."""
    believed = np.asarray(cube.global_pose.to_np())[:3, 3].ravel()
    actual = np.array(cube_sensor.position())
    print(f'{label:14s} twin {np.round(believed, 3)}  '
          f'sim {np.round(actual, 3)}  gap {np.linalg.norm(believed - actual):.3f} m')
    return believed, actual


cube_status('start')

start          twin [ 0.45  -0.2    0.325]  sim [ 0.45  -0.2    0.325]  gap 0.000 m


(array([ 0.45000008, -0.19999997,  0.32499999]),
 array([ 0.45000008, -0.19999997,  0.32499999]))

## 2. Park the arm and open the hand

`ParkArmsAction` looks up the Panda's park configuration (Franka's own "ready"
pose) and `SetGripperAction` its open finger travel, both from the semantic
model. Unlike the Stretch, no widening is needed: the hand's `GripperState.OPEN`
is 0.038 m of travel per finger, i.e. **0.076 m between the pads** against a
0.05 m cube — 1.3 cm of clearance either side, which is how much approach error
the grasp tolerates before a finger knocks the cube off its stand.

In [6]:
from coraplex.datastructures.enums import Arms
from coraplex.robot_plans.actions.core.robot_body import ParkArmsAction, SetGripperAction
from semantic_digital_twin.datastructures.definitions import GripperState

from cram_vrb_lab.robots.panda.joints import FINGER_JOINTS
from cram_vrb_lab.robots.panda.semantic_model import gripper_pad_gap

run_plan(execute_single(ParkArmsAction(Arms.LEFT), context=context))
run_plan(execute_single(SetGripperAction(Arms.LEFT, GripperState.OPEN), context=context))

travel = world.get_connection_by_name(FINGER_JOINTS[0]).position
print(f'fingers at {travel:.3f} m travel = {gripper_pad_gap(travel):.3f} m between '
      f'the pads (needs to clear the {props.CUBE_SIZE:.3f} m cube)')

ImportError: cannot import name 'gripper_pad_gap' from 'cram_vrb_lab.robots.panda.semantic_model' (/home/jovyan/cram-vrb-lab/cram_vrb_lab/robots/panda/semantic_model.py)

`Arms.LEFT` is not a claim about handedness — the Panda has one arm, and CRAM's
`ViewManager` hands back the only arm there is whatever you ask for. It just has
to be *some* member of the enum.

## 3. Grasp the cube

`PickUpAction` expands to *open gripper → move to a pre-grasp pose → move to the
grasp pose → close gripper → attach in the twin → lift*. With
`FRONT` + `TOP` the three tool poses are, in map coordinates:

| | position | hand points |
|---|---|---|
| pre-grasp | (0.45, −0.20, 0.400) | down |
| grasp | (0.45, −0.20, 0.325) | down |
| lift | (0.45, −0.20, 0.375) | down |

i.e. straight down onto the cube and straight back up, 7.5 cm of clearance on
the way in (the cube's half-height plus `GraspDescription.manipulation_offset`).

`collision_avoidance=False`: the hand has to get *inside* the avoidance margin of
the cube and the stand, which is exactly what the margin forbids.

**Why the close-gripper step needs a Panda-specific motion.** CRAM turns a
gripper state into a joint-position goal on the fingers, and under closed-loop
control that goal is checked against the *measured* finger positions. A hand
closing on a rigid object never reaches them — the cube stops the fingers 2.5 cm
short of closed — so the default 1 cm tolerance is never met and the grasp hangs
right there, holding the cube, looking for all the world like it worked.
`PandaMoveGripper` (in `cram_vrb_lab/robots/panda/motions.py`) gives *closing*
a 3 cm tolerance. Opening keeps the tight one; nothing obstructs it.

Ending the task is not the same as letting go: the fingers keep driving towards a
target that now lies inside the cube, so they go on squeezing through the lift
and the transfer.

We re-sync from the sim first, so the grasp is planned against where the cube
really is rather than where it was spawned.

In [ ]:
from coraplex.datastructures.enums import ApproachDirection, VerticalAlignment
from coraplex.datastructures.grasp import GraspDescription
from coraplex.robot_plans.actions.core.pick_up import PickUpAction
from coraplex.view_manager import ViewManager

grasp = GraspDescription(
    ApproachDirection.FRONT,   # with TOP below: approach along map -z
    VerticalAlignment.TOP,
    ViewManager.get_end_effector_view(Arms.LEFT, robot),
)

sync_cube_from_sim(world, cube, cube_sensor)
run_plan(
    execute_single(PickUpAction(cube, Arms.LEFT, grasp), context=context),
    collision_avoidance=False,
)
cube_status('after grasp')

**If the cube did not come along** (the sim pose is still on the stand at
z ≈ 0.325 while the twin pose has risen), in the order worth checking:

- *The fingers closed beside the cube.* Watch the Isaac viewport during the
  descent. A lateral miss means the twin's cube pose and the rendered one had
  drifted — re-run the `sync_cube_from_sim` cell and try again.
- *The fingers closed but the cube squirted out.* Raise `CUBE_FRICTION` in
  `cram_vrb_lab/scenes/props/constants.py`, or lower `CUBE_MASS`.
- *The fingers never really closed.* The streamed velocity target may lead the
  measured position by `StreamedVelocityIntegrator.MAX_LEAD` (0.02 m), so the
  grip force is `FINGER_DRIVE_STIFFNESS × 0.02`. Both live in
  `cram_vrb_lab/`; raise the stiffness if the fingers stall on contact.

## 4. Put it down on the other stand

`PlaceAction` is the mirror of the pick: reach the target through the grasp
sequence in reverse, open the gripper, detach in the twin, retract. The whole
transfer is 0.4 m of arm motion — no base involved.

In [ ]:
from coraplex.robot_plans.actions.core.placing import PlaceAction
from semantic_digital_twin.spatial_types import Point3
from semantic_digital_twin.spatial_types.spatial_types import Pose

place_target = Pose(
    Point3.from_iterable(PANDA_LAYOUT.cube_target_position),
    reference_frame=world.root,
)
run_plan(
    execute_single(PlaceAction(cube, place_target, Arms.LEFT), context=context),
    collision_avoidance=False,
)
run_plan(execute_single(ParkArmsAction(Arms.LEFT), context=context))
cube_status('after place')

## 5. Verdict

The cube's true resting position against where it was asked to go. Both parts
matter: the height says it is on a stand rather than the floor, and the x/y error
says it is on the *right* stand — a cube never picked up would score a perfect
height and be 0.4 m out in y.

A validated run of this notebook ends with an x/y error of about 3 mm.

In [ ]:
_, actual = cube_status('final')
target = np.array(PANDA_LAYOUT.cube_target_position)
error = actual - target
placed = np.linalg.norm(error[:2]) < 0.05 and abs(error[2]) < 0.02
print(f'target {np.round(target, 3)}')
print(f'error  {np.round(error, 3)}   |xy| {np.linalg.norm(error[:2]):.3f} m')
print('PLACED on the stand' if placed else 'NOT placed -- see cube_status above')

## The same thing as one designator

`TransportAction` is the fetch-and-carry composite: grasp, lift, carry, place —
one designator. On a mobile robot it also resolves standing positions; here there
is nothing to resolve, so it is purely a shorthand.

Worth running once the step-by-step version works. To retry it, restart the sim
so the cube is back on the pick stand.

In [ ]:
# from coraplex.robot_plans.actions.composite.transporting import TransportAction
#
# sync_cube_from_sim(world, cube, cube_sensor)
# run_plan(
#     execute_single(TransportAction(cube, place_target, Arms.LEFT), context=context),
#     collision_avoidance=False,
# )
# cube_status('after transport')

## Shutdown

In [ ]:
stop(patterns=(PANDA_SIM_SCRIPT.name, PANDA_SERVER_SCRIPT.name))